In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import yaml
import pandas as pd
import numpy as np

import torch
from model import LightningWBoson
from physics.torchBoost import Booster as TorchBooster
from physics.physics import dphi as dphi
from physics.physics import sphi as sphi
from train import train

from matplotlib import pyplot as plt
from notebooks.plottingtool import plot_1d_hist, plot_2d_hist, plot_2d_res_hist

import mplhep as hep
hep.style.use("ATLAS")


In [ ]:
CONFIG_PATH = REPO_ROOT / "configs/config.yaml"
with CONFIG_PATH.open() as file:
    cfg = yaml.safe_load(file)

# Visualize the configured training output, selecting the best checkpoint when its metric is in the filename.
RUN_DIR = Path(cfg["paths"]["saved_path"]).expanduser()
if not RUN_DIR.is_absolute():
    RUN_DIR = (REPO_ROOT / RUN_DIR).resolve()

log_dirs = sorted((RUN_DIR / "logs").glob("version_*"), key=lambda path: path.stat().st_mtime, reverse=True)
ckpt_files = sorted((RUN_DIR / "logs").glob("version_*/checkpoints/*.ckpt"), key=lambda path: path.stat().st_mtime, reverse=True)
print(f"Training run directory: {RUN_DIR}")
print(f"Found log directories: {[str(path) for path in log_dirs]}")
print(f"Found checkpoint files: {[str(path) for path in ckpt_files]}")
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found under {RUN_DIR}")

trainer_cfg = cfg.get("trainer", {})
monitor_metric = trainer_cfg.get("monitor_metric", "val_static_loss")
monitor_mode = trainer_cfg.get("monitor_mode", "min")
ranked_ckpts = [path for path in ckpt_files if path.name != "last.ckpt"] or ckpt_files
metric_candidates = [monitor_metric, "val_static_loss", "val_loss"]
parsed_ckpts = []
selected_metric = None
for metric_name in metric_candidates:
    token = f"{metric_name}="
    parsed_ckpts = []
    for path in ranked_ckpts:
        if token not in path.stem:
            continue
        value_text = path.stem.rsplit(token, 1)[1].split("-", 1)[0]
        try:
            parsed_ckpts.append((path, float(value_text)))
        except ValueError:
            pass
    if parsed_ckpts:
        selected_metric = metric_name
        break

if parsed_ckpts:
    ckpt_path, ckpt_metric_value = sorted(parsed_ckpts, key=lambda item: item[1], reverse=(monitor_mode == "max"))[0]
    print(f"Using best checkpoint by {selected_metric}: {ckpt_path} ({ckpt_metric_value:.4g})")
else:
    ckpt_path = ranked_ckpts[0]
    print(f"Using most recent checkpoint: {ckpt_path}")

LOG_DIR = ckpt_path.parents[1]
print(f"Using log directory: {LOG_DIR}")


In [ ]:
# Retrieve the same data split definition used by the original training config.
dm = train.main(train=False, config_path=str(CONFIG_PATH))
dm.setup(stage="test")
test_loader = dm.test_dataloader()

model = LightningWBoson.load_from_checkpoint(str(ckpt_path), weights_only=False, strict=False)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

feature_batches = []
prediction_batches = []
dmet_batches = []
target_batches = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        outputs, aux = model(inputs, return_aux=True)

        feature_batches.append(inputs.cpu().numpy())
        prediction_batches.append(outputs.cpu().numpy())
        dmet_batches.append(aux["dmet"].cpu().numpy())
        target_batches.append(targets.cpu().numpy())

train_features = np.concatenate(feature_batches, axis=0)
predictions = np.concatenate(prediction_batches, axis=0)
pred_dmet = np.concatenate(dmet_batches, axis=0)
true_labels = np.concatenate(target_batches, axis=0)
print(f"Evaluated {len(predictions)} test events on {device}.")


In [ ]:
true_lep0_px = train_features[:, 0]
true_lep0_py = train_features[:, 1]
true_lep0_pz = train_features[:, 2]
true_lep0_energy = train_features[:, 3]
true_lep1_px = train_features[:, 4]
true_lep1_py = train_features[:, 5]
true_lep1_pz = train_features[:, 6]
true_lep1_energy = train_features[:, 7]
true_met_px = train_features[:, 16]
true_met_py = train_features[:, 17]
true_met_pt = np.sqrt(true_met_px**2 + true_met_py**2)

In [ ]:
pred_w0_px = predictions[..., 0]
pred_w0_py = predictions[..., 1]
pred_w0_pz = predictions[..., 2]
pred_w0_energy = predictions[..., 3]
pred_w1_px = predictions[..., 4]
pred_w1_py = predictions[..., 5]
pred_w1_pz = predictions[..., 6]
pred_w1_energy = predictions[..., 7]
pred_w0_mass_2 = pred_w0_energy**2 - (pred_w0_px**2 + pred_w0_py**2 + pred_w0_pz**2)
pred_w1_mass_2 = pred_w1_energy**2 - (pred_w1_px**2 + pred_w1_py**2 + pred_w1_pz**2)

true_w0_px = true_labels[..., 0]
true_w0_py = true_labels[..., 1]
true_w0_pz = true_labels[..., 2]
true_w0_energy = true_labels[..., 3]
true_w1_px = true_labels[..., 4]
true_w1_py = true_labels[..., 5]
true_w1_pz = true_labels[..., 6]
true_w1_energy = true_labels[..., 7]
true_w0_mass_2 = true_labels[..., 8] ** 2
true_w1_mass_2 = true_labels[..., 9] ** 2

In [ ]:
pred_nu0_px = pred_w0_px - true_lep0_px
pred_nu0_py = pred_w0_py - true_lep0_py
pred_nu0_pz = pred_w0_pz - true_lep0_pz

pred_nu1_px = pred_w1_px - true_lep1_px
pred_nu1_py = pred_w1_py - true_lep1_py
pred_nu1_pz = pred_w1_pz - true_lep1_pz

pred_dinu_px = pred_nu0_px + pred_nu1_px
pred_dinu_py = pred_nu0_py + pred_nu1_py
pred_dinu_pt = np.sqrt(pred_dinu_px**2 + pred_dinu_py**2)


true_nu0_px = true_w0_px - true_lep0_px
true_nu0_py = true_w0_py - true_lep0_py
true_nu0_pz = true_w0_pz - true_lep0_pz

true_nu1_px = true_w1_px - true_lep1_px
true_nu1_py = true_w1_py - true_lep1_py
true_nu1_pz = true_w1_pz - true_lep1_pz

true_dinu_px = true_nu0_px + true_nu1_px
true_dinu_py = true_nu0_py + true_nu1_py
true_dinu_pt = np.sqrt(true_dinu_px**2 + true_dinu_py**2)

pred_dmet_px = pred_dmet[:, 0]
pred_dmet_py = pred_dmet[:, 1]
pred_dmet_pt = np.sqrt(pred_dmet_px**2 + pred_dmet_py**2)

true_dmet_px = true_met_px - true_dinu_px
true_dmet_py = true_met_py - true_dinu_py
true_dmet_pt = np.sqrt(true_dmet_px**2 + true_dmet_py**2)

pred_neutrino_met_px = true_met_px - pred_dmet_px
pred_neutrino_met_py = true_met_py - pred_dmet_py
pred_neutrino_met_pt = np.sqrt(pred_neutrino_met_px**2 + pred_neutrino_met_py**2)

In [ ]:
bins = np.linspace(-100, 100, 51)
plot_1d_hist(pred_dmet_px, true_dmet_px, r"$\Delta E_{T,x}^{miss}$", bins_edges=bins, unit="GeV")
plot_2d_hist(pred_dmet_px, true_dmet_px, r"$\Delta E_{T,x}^{miss}$", bins_edges=bins, log=True, unit="GeV", vmax=2e3)
plot_1d_hist(pred_dmet_py, true_dmet_py, r"$\Delta E_{T,y}^{miss}$", bins_edges=bins, unit="GeV")
plot_2d_hist(pred_dmet_py, true_dmet_py, r"$\Delta E_{T,y}^{miss}$", bins_edges=bins, log=True, unit="GeV", vmax=2e3)

pt_bins = np.linspace(0, 150, 51)
plot_1d_hist(pred_dmet_pt, true_dmet_pt, r"$|\Delta E_T^{miss}|$", bins_edges=pt_bins, unit="GeV")
plot_2d_hist(pred_dmet_pt, true_dmet_pt, r"$|\Delta E_T^{miss}|$", bins_edges=pt_bins, log=True, unit="GeV", vmax=2e3)

In [ ]:
plot_1d_hist(pred_dinu_pt, true_dinu_pt, r"$p_T^{\nu\nu}$", bins_edges=np.linspace(0, 200, 51), unit="GeV")
plot_2d_hist(pred_dinu_pt, true_dinu_pt, r"$p_T^{\nu\nu}$", bins_edges=np.linspace(0, 200, 51), log=True, unit="GeV", vmax=2e3)

In [ ]:
def safe_mass(mass2):
    return np.sqrt(np.abs(mass2))

pred_higgs_px = pred_w0_px + pred_w1_px
pred_higgs_py = pred_w0_py + pred_w1_py
pred_higgs_pz = pred_w0_pz + pred_w1_pz
pred_higgs_pt = np.sqrt(pred_higgs_px**2 + pred_higgs_py**2)
pred_higgs_energy = pred_w0_energy + pred_w1_energy
pred_higgs_mass_2 = pred_higgs_energy**2 - (pred_higgs_px**2 + pred_higgs_py**2 + pred_higgs_pz**2)
pred_higgs_mass = safe_mass(pred_higgs_mass_2)

true_higgs_px = true_w0_px + true_w1_px
true_higgs_py = true_w0_py + true_w1_py
true_higgs_pz = true_w0_pz + true_w1_pz
true_higgs_pt = np.sqrt(true_higgs_px**2 + true_higgs_py**2)
true_higgs_energy = true_w0_energy + true_w1_energy
true_higgs_mass_2 = true_higgs_energy**2 - (true_higgs_px**2 + true_higgs_py**2 + true_higgs_pz**2)
true_higgs_mass = safe_mass(true_higgs_mass_2)

# plot_1d_hist(pred_higgs_px, true_higgs_px, "$p_x^{H}$", bins_edges=np.linspace(-200, 200, 51), unit="GeV")
# plot_2d_hist(pred_higgs_px, true_higgs_px, "$p_x^{H}$", bins_edges=np.linspace(-200, 200, 51), log=True, unit="GeV", vmax=2e3)
# plot_1d_hist(pred_higgs_py, true_higgs_py, "$p_y^{H}$", bins_edges=np.linspace(-200, 200, 51), unit="GeV")
# plot_2d_hist(pred_higgs_py, true_higgs_py, "$p_y^{H}$", bins_edges=np.linspace(-200, 200, 51), log=True, unit="GeV", vmax=2e3)
plot_1d_hist(pred_higgs_pt, true_higgs_pt, "$p_T^{H}$", bins_edges=np.linspace(0, 600, 51), unit="GeV")
plot_2d_hist(pred_higgs_pt, true_higgs_pt, "$p_T^{H}$", bins_edges=np.linspace(0, 600, 51), log=True, unit="GeV", vmax=2e3)
plot_1d_hist(pred_higgs_pz, true_higgs_pz, "$p_z^{H}$", bins_edges=np.linspace(-600, 600, 51), unit="GeV")
plot_2d_hist(pred_higgs_pz, true_higgs_pz, "$p_z^{H}$", bins_edges=np.linspace(-600, 600, 51), log=True, unit="GeV", vmax=2e3)
plot_1d_hist(pred_higgs_energy, true_higgs_energy, "$E_H$", bins_edges=np.linspace(100, 600, 51), unit="GeV")
plot_2d_hist(pred_higgs_energy, true_higgs_energy, "$E_H$", bins_edges=np.linspace(100, 600, 51), log=True, unit="GeV", vmax=2e3)
plot_1d_hist(pred_higgs_mass, true_higgs_mass, "$m_H$", bins_edges=np.linspace(120, 130, 51), unit="GeV")
print(f"Mean predicted Higgs mass: {pred_higgs_mass.mean():.2f} GeV, Std: {pred_higgs_mass.std():.2f} GeV")
plot_2d_hist(pred_higgs_mass, true_higgs_mass, "$m_H$", bins_edges=np.linspace(120, 130, 51), log=True, unit="GeV", vmax=2e3)


In [ ]:
bins = np.linspace(-150, 150, 51)
plot_1d_hist(pred_w0_px, true_w0_px, bins_edges=bins, name=r"$p_x^{W^+}$")
plot_2d_hist(pred_w0_px, true_w0_px, name=r"$p_x^{W^+}$", bins_edges=bins, log=True, vmax=2e3)
plot_1d_hist(pred_w1_px, true_w1_px, bins_edges=bins, name=r"$p_x^{W^-}$")
plot_2d_hist(pred_w1_px, true_w1_px, name=r"$p_x^{W^-}$", bins_edges=bins, log=True, vmax=2e3)

In [ ]:
bins = np.linspace(-150, 150, 51)
plot_1d_hist(pred_w0_py, true_w0_py, bins_edges=bins, name=r"$p_y^{W^+}$")
plot_2d_hist(pred_w0_py, true_w0_py, name=r"$p_y^{W^+}$", bins_edges=bins, log=True, vmax=2e3)
plot_1d_hist(pred_w1_py, true_w1_py, bins_edges=bins, name=r"$p_y^{W^-}$")
plot_2d_hist(pred_w1_py, true_w1_py, name=r"$p_y^{W^-}$", bins_edges=bins, log=True, vmax=2e3)

In [ ]:
bins = np.linspace(-300, 300, 51)
plot_1d_hist(pred_w0_pz, true_w0_pz, bins_edges=bins, name=r"$p_z^{W^+}$")
plot_2d_hist(pred_w0_pz, true_w0_pz, name=r"$p_z^{W^+}$", bins_edges=bins, log=True, vmax=2e3)
plot_1d_hist(pred_w1_pz, true_w1_pz, bins_edges=bins, name=r"$p_z^{W^-}$")
plot_2d_hist(pred_w1_pz, true_w1_pz, name=r"$p_z^{W^-}$", bins_edges=bins, log=True, vmax=2e3)

In [ ]:
bins = np.linspace(0, 300, 51)
plot_1d_hist(pred_w0_energy, true_w0_energy, bins_edges=bins, name=r"$E^{W^+}$")
plot_2d_hist(pred_w0_energy, true_w0_energy, name=r"$E^{W^+}$", bins_edges=bins, log=True, color="black", vmax=2e3)
plot_1d_hist(pred_w1_energy, true_w1_energy, bins_edges=bins, name=r"$E^{W^-}$")
plot_2d_hist(pred_w1_energy, true_w1_energy, name=r"$E^{W^-}$", bins_edges=bins, log=True, color="black", vmax=2e3)

In [ ]:
bins = np.linspace(0, 100, 51)
pred_w0_mass = safe_mass(pred_w0_mass_2)
pred_w1_mass = safe_mass(pred_w1_mass_2)
true_w0_mass = safe_mass(true_w0_mass_2)
true_w1_mass = safe_mass(true_w1_mass_2)

plot_1d_hist(pred_w0_mass, true_w0_mass, bins_edges=bins, name=r"$m_{W^+}$", unit="GeV")
plot_2d_hist(pred_w0_mass, true_w0_mass, name=r"$m_{W^+}$", bins_edges=bins, log=True, unit="GeV", vmax=2e3)
plot_1d_hist(pred_w1_mass, true_w1_mass, bins_edges=bins, name=r"$m_{W^-}$", unit="GeV")
plot_2d_hist(pred_w1_mass, true_w1_mass, name=r"$m_{W^-}$", bins_edges=bins, log=True, unit="GeV", vmax=2e3)


In [ ]:
# bins = np.linspace(0, 100, 51)

# # W0: infer the neutrino from the predicted W and test the swapped lepton.
# diff_w0 = pred_w0_mass - true_w0_mass
# pred_nu0_px = pred_w0_px - true_lep0_px
# pred_nu0_py = pred_w0_py - true_lep0_py
# pred_nu0_pz = pred_w0_pz - true_lep0_pz
# pred_nu0_energy = pred_w0_energy - true_lep0_energy
# pred_nu0_mass_2 = (
#     pred_nu0_energy**2
#     - (pred_nu0_px**2 + pred_nu0_py**2 + pred_nu0_pz**2)
# )
# pred_nu0_mass = safe_mass(pred_nu0_mass_2)

# pred_w0_px_swap = pred_nu0_px + true_lep1_px
# pred_w0_py_swap = pred_nu0_py + true_lep1_py
# pred_w0_pz_swap = pred_nu0_pz + true_lep1_pz
# pred_w0_energy_swap = pred_nu0_energy + true_lep1_energy
# pred_w0_mass_swap = safe_mass(
#     pred_w0_energy_swap**2
#     - (pred_w0_px_swap**2 + pred_w0_py_swap**2 + pred_w0_pz_swap**2)
# )
# diff_w0_swap = pred_w0_mass_swap - true_w0_mass
# corr_mask_w0 = np.abs(diff_w0_swap) < np.abs(diff_w0)
# corr_pred_w0_px = np.where(corr_mask_w0, pred_w0_px_swap, pred_w0_px)
# corr_pred_w0_py = np.where(corr_mask_w0, pred_w0_py_swap, pred_w0_py)
# corr_pred_w0_pz = np.where(corr_mask_w0, pred_w0_pz_swap, pred_w0_pz)
# corr_pred_w0_energy = np.where(corr_mask_w0, pred_w0_energy_swap, pred_w0_energy)
# corr_pred_w0_p4 = np.stack([corr_pred_w0_px, corr_pred_w0_py, corr_pred_w0_pz, corr_pred_w0_energy], axis=1)
# corr_pred_w0_mass = np.where(corr_mask_w0, pred_w0_mass_swap, pred_w0_mass)

# # W1: mirror the same correction using the opposite lepton assignment.
# diff_w1 = pred_w1_mass - true_w1_mass
# pred_nu1_px = pred_w1_px - true_lep1_px
# pred_nu1_py = pred_w1_py - true_lep1_py
# pred_nu1_pz = pred_w1_pz - true_lep1_pz
# pred_nu1_energy = pred_w1_energy - true_lep1_energy
# pred_nu1_mass_2 = (
#     pred_nu1_energy**2
#     - (pred_nu1_px**2 + pred_nu1_py**2 + pred_nu1_pz**2)
# )
# pred_nu1_mass = safe_mass(pred_nu1_mass_2)

# pred_w1_px_swap = pred_nu1_px + true_lep0_px
# pred_w1_py_swap = pred_nu1_py + true_lep0_py
# pred_w1_pz_swap = pred_nu1_pz + true_lep0_pz
# pred_w1_energy_swap = pred_nu1_energy + true_lep0_energy
# pred_w1_mass_swap = safe_mass(
#     pred_w1_energy_swap**2
#     - (pred_w1_px_swap**2 + pred_w1_py_swap**2 + pred_w1_pz_swap**2)
# )
# diff_w1_swap = pred_w1_mass_swap - true_w1_mass
# corr_mask_w1 = np.abs(diff_w1_swap) < np.abs(diff_w1)
# corr_pred_w1_px = np.where(corr_mask_w1, pred_w1_px_swap, pred_w1_px)
# corr_pred_w1_py = np.where(corr_mask_w1, pred_w1_py_swap, pred_w1_py)
# corr_pred_w1_pz = np.where(corr_mask_w1, pred_w1_pz_swap, pred_w1_pz)
# corr_pred_w1_energy = np.where(corr_mask_w1, pred_w1_energy_swap, pred_w1_energy)
# corr_pred_w1_p4 = np.stack([corr_pred_w1_px, corr_pred_w1_py, corr_pred_w1_pz, corr_pred_w1_energy], axis=1)
# corr_pred_w1_mass = np.where(corr_mask_w1, pred_w1_mass_swap, pred_w1_mass)

# # True neutrino masses for reference plots.
# true_nu0_px = true_w0_px - true_lep0_px
# true_nu0_py = true_w0_py - true_lep0_py
# true_nu0_pz = true_w0_pz - true_lep0_pz
# true_nu0_energy = true_w0_energy - true_lep0_energy
# true_nu0_mass = safe_mass(
#     true_nu0_energy**2
#     - (true_nu0_px**2 + true_nu0_py**2 + true_nu0_pz**2)
# )

# true_nu1_px = true_w1_px - true_lep1_px
# true_nu1_py = true_w1_py - true_lep1_py
# true_nu1_pz = true_w1_pz - true_lep1_pz
# true_nu1_energy = true_w1_energy - true_lep1_energy
# true_nu1_mass = safe_mass(
#     true_nu1_energy**2
#     - (true_nu1_px**2 + true_nu1_py**2 + true_nu1_pz**2)
# )

# plot_1d_hist(corr_pred_w0_mass, true_w0_mass, bins_edges=bins, name=r"Corrected $m_{W^+}$", unit="GeV")
# plot_2d_hist(corr_pred_w0_mass, true_w0_mass, name=r"Corrected $m_{W^+}$", bins_edges=bins, log=True, unit="GeV", vmax=2e3)
# plot_1d_hist(corr_pred_w1_mass, true_w1_mass, bins_edges=bins, name=r"Corrected $m_{W^-}$", unit="GeV")
# plot_2d_hist(corr_pred_w1_mass, true_w1_mass, name=r"Corrected $m_{W^-}$", bins_edges=bins, log=True, unit="GeV", vmax=2e3)


In [ ]:
# diff_mass_pos = pred_w0_mass - true_w0_mass
# diff_mass_neg = pred_w1_mass - true_w1_mass
# bins_edges = np.linspace(-100, 100, 51)
# plt.hist(diff_mass_pos, bins=bins_edges, histtype='step', color='red',label=r"$\Delta m_{W^+}$")
# plt.hist(diff_mass_neg, bins=bins_edges, histtype='step', color='blue',label=r"$\Delta m_{W^-}$")
# plt.xlabel(r"$\Delta m_W$ [GeV]")
# plt.ylabel("Events")
# plt.legend()
# plt.show()

In [ ]:
w0_mass_res = pred_w0_mass - true_w0_mass
w1_res = pred_w1_mass - true_w1_mass
plot_2d_res_hist(w0_mass_res, w1_res, name_pos=r"$m_{W^+}$", name_neg=r"$m_{W^-}$", bins_edges=np.linspace(-80, 80, 51), log=True, unit="GeV", color="black", vmax=2e3)

w0_px_res = pred_w0_px - true_w0_px
w1_px_res = pred_w1_px - true_w1_px
plot_2d_res_hist(w0_px_res, w1_px_res, name_pos=r"$p_x^{W^+}$", name_neg=r"$p_x^{W^-}$", bins_edges=np.linspace(-100, 100, 51), log=True, unit="GeV", color="black", vmax=2e3)

w0_pz_res = pred_w0_pz - true_w0_pz
w1_pz_res = pred_w1_pz - true_w1_pz
plot_2d_res_hist(w0_pz_res, w1_pz_res, name_pos=r"$p_z^{W^+}$", name_neg=r"$p_z^{W^-}$", bins_edges=np.linspace(-800, 800, 51), log=True, unit="GeV", color="black", vmax=2e3)

w0_energy_res = pred_w0_energy - true_w0_energy
w1_energy_res = pred_w1_energy - true_w1_energy
plot_2d_res_hist(w0_energy_res, w1_energy_res, name_pos=r"$E^{W^+}$", name_neg=r"$E^{W^-}$", bins_edges=np.linspace(-1000, 500, 51), log=True, unit="GeV", color="black", vmax=2e3)

In [ ]:
def alpha_func(lep_on_p4, neu_on_p4, lep_off_p4, neu_off_p4):
    
    def _cal_mass(p4):
        return np.sqrt(p4[:, 3]**2 - p4[:, 0]**2 - p4[:, 1]**2 - p4[:, 2]**2)
    
    def _cal_norm(p4):
        return np.sqrt(p4[:, 0]**2 + p4[:, 1]**2 + p4[:, 2]**2)
    
    # dinu_p4 = neu_on_p4 + neu_off_p4
    # dinu_mass = _cal_mass(dinu_p4)
    dinu_lep_on_p4 = lep_on_p4 + neu_on_p4 + neu_off_p4
    dinu_lep_on_mass = _cal_mass(dinu_lep_on_p4)
    dinu_lep_off_p4 = lep_off_p4 + neu_on_p4 + neu_off_p4
    dinu_lep_off_mass = _cal_mass(dinu_lep_off_p4)
        
    alpha = np.where(
        dinu_lep_on_mass > dinu_lep_off_mass,
        _cal_norm(neu_on_p4) / (_cal_norm(neu_on_p4) + _cal_norm(neu_off_p4) + 1e-16),
        _cal_norm(neu_off_p4) / (_cal_norm(neu_on_p4) + _cal_norm(neu_off_p4) + 1e-16)
    )
    
    return alpha

# true
true_w0_p4 = np.stack([true_w0_px, true_w0_py, true_w0_pz, true_w0_energy], axis=-1)
true_w1_p4 = np.stack([true_w1_px, true_w1_py, true_w1_pz, true_w1_energy], axis=-1)
true_lep0_p4 = np.stack([true_lep0_px, true_lep0_py, true_lep0_pz, true_lep0_energy], axis=-1)
true_lep1_p4 = np.stack([true_lep1_px, true_lep1_py, true_lep1_pz, true_lep1_energy], axis=-1)

# pred
pred_w0_p4 = np.stack([pred_w0_px, pred_w0_py, pred_w0_pz, pred_w0_energy], axis=-1)
pred_w1_p4 = np.stack([pred_w1_px, pred_w1_py, pred_w1_pz, pred_w1_energy], axis=-1)

w_mass_con = 80.379 # GeV
w_mass_con_2 = w_mass_con ** 2

true_0_on_mask = np.abs(true_w0_mass_2 - w_mass_con_2) < np.abs(true_w1_mass_2 - w_mass_con_2)
true_lep_on_p4 = np.where(true_0_on_mask[:, np.newaxis], true_lep0_p4, true_lep1_p4)
true_lep_off_p4 = np.where(true_0_on_mask[:, np.newaxis], true_lep1_p4, true_lep0_p4)
true_w_on_p4 = np.where(true_0_on_mask[:, np.newaxis], true_w0_p4, true_w1_p4)
true_w_off_p4 = np.where(true_0_on_mask[:, np.newaxis], true_w1_p4, true_w0_p4)
true_neu_on_p4 = true_w_on_p4 - true_lep_on_p4
true_neu_off_p4 = true_w_off_p4 - true_lep_off_p4

pred_0_on_mask = np.abs(pred_w0_mass_2 - w_mass_con_2) < np.abs(pred_w1_mass_2 - w_mass_con_2)
pred_w_on_p4 = np.where(pred_0_on_mask[:, np.newaxis], pred_w0_p4, pred_w1_p4)
pred_w_off_p4 = np.where(pred_0_on_mask[:, np.newaxis], pred_w1_p4, pred_w0_p4)
pred_neu_on_p4 = pred_w_on_p4 - true_lep_on_p4
pred_neu_off_p4 = pred_w_off_p4 - true_lep_off_p4

true_alpha = alpha_func(true_lep_on_p4, true_neu_on_p4, true_lep_off_p4, true_neu_off_p4)
pred_alpha = alpha_func(true_lep_on_p4, pred_neu_on_p4, true_lep_off_p4, pred_neu_off_p4)

In [ ]:
plot_1d_hist(pred_alpha, true_alpha, r"$\alpha$", np.linspace(0, 1, 51), unit="null")
plot_2d_hist(pred_alpha, true_alpha, r"$\alpha$", np.linspace(0, 1, 51), log=True, unit="null", color="black", vmax=1e3)

In [ ]:
lep_p4 = np.concatenate([true_lep0_p4, true_lep1_p4], axis=1)
true_w_p4 = np.concatenate([true_w0_p4, true_w1_p4], axis=1)
pred_w_p4 = np.concatenate([pred_w0_p4, pred_w1_p4], axis=1)

def angular_features_like_training(lep_p4, true_w_p4, pred_w_p4, device):
    lep = torch.as_tensor(lep_p4, dtype=torch.float32, device=device)
    true_w = torch.as_tensor(true_w_p4, dtype=torch.float32, device=device)
    pred_w = torch.as_tensor(pred_w_p4, dtype=torch.float32, device=device)

    true_booster = TorchBooster(lep, true_w)
    pred_booster = TorchBooster(lep, pred_w)
    valid = true_booster.valid_rest_frame_mask() & pred_booster.valid_rest_frame_mask()

    true_ang = torch.stack(true_booster.lep_theta_phi_in_w_rest(), dim=-1)[valid]
    pred_ang = torch.stack(pred_booster.lep_theta_phi_in_w_rest(), dim=-1)[valid]
    return true_ang.cpu().numpy(), pred_ang.cpu().numpy(), valid.cpu().numpy()

true_ang, pred_ang, angular_valid = angular_features_like_training(lep_p4, true_w_p4, pred_w_p4, device)
print(f"Angular valid events: {angular_valid.sum()} / {len(angular_valid)}")

# true_l0_theta_phi = (true_ang[:, 0], true_ang[:, 1], true_ang[:, 2])
# true_l1_theta_phi = (true_ang[:, 3], true_ang[:, 4], true_ang[:, 5])
# pred_l0_theta_phi = (pred_ang[:, 0], pred_ang[:, 1], pred_ang[:, 2])
# pred_l1_theta_phi = (pred_ang[:, 3], pred_ang[:, 4], pred_ang[:, 5])
true_l0_theta_phi = (true_ang[:, 0], true_ang[:, 1])
true_l1_theta_phi = (true_ang[:, 2], true_ang[:, 3])
pred_l0_theta_phi = (pred_ang[:, 0], pred_ang[:, 1])
pred_l1_theta_phi = (pred_ang[:, 2], pred_ang[:, 3])


In [ ]:
plot_1d_hist(pred_l0_theta_phi[0] / np.pi, true_l0_theta_phi[0] / np.pi, r"$\theta^\ast_{\ell^+}$", np.linspace(0, 1, 51), unit="rad/$\pi$")
plot_2d_hist(pred_l0_theta_phi[0] / np.pi, true_l0_theta_phi[0] / np.pi, r"$\theta^\ast_{\ell^+}$", np.linspace(0, 1, 51), log=True, unit="rad/$\pi$", color="black", vmax=8e2)
plot_1d_hist(pred_l1_theta_phi[0] / np.pi, true_l1_theta_phi[0] / np.pi, r"$\theta^\ast_{\ell^-}$", np.linspace(0, 1, 51), unit="rad/$\pi$")
plot_2d_hist(pred_l1_theta_phi[0] / np.pi, true_l1_theta_phi[0] / np.pi, r"$\theta^\ast_{\ell^-}$", np.linspace(0, 1, 51), log=True, unit="rad/$\pi$", color="black", vmax=8e2)

In [ ]:
# plot_1d_hist(pred_l0_theta_phi[1], true_l0_theta_phi[1], r"$\sin(\phi^\ast_{\ell^+})$", np.linspace(-1, 1, 61), unit="null")
# plot_2d_hist(pred_l0_theta_phi[1], true_l0_theta_phi[1], r"$\sin(\phi^\ast_{\ell^+})$", np.linspace(-1, 1, 61), log=True, unit="null", color="black", vmax=1e2)
# plot_1d_hist(pred_l0_theta_phi[2], true_l0_theta_phi[2], r"$\cos(\phi^\ast_{\ell^+})$", np.linspace(-1, 1, 61), unit="null")
# plot_2d_hist(pred_l0_theta_phi[2], true_l0_theta_phi[2], r"$\cos(\phi^\ast_{\ell^+})$", np.linspace(-1, 1, 61), log=True, unit="null", color="black", vmax=1e2)

In [ ]:
# plot_1d_hist(pred_l1_theta_phi[1], true_l1_theta_phi[1], r"$\sin(\phi^\ast_{\ell^-})$", np.linspace(-1, 1, 61), unit="null")
# plot_2d_hist(pred_l1_theta_phi[1], true_l1_theta_phi[1], r"$\sin(\phi^\ast_{\ell^-})$", np.linspace(-1, 1, 61), log=True, unit="null", color="black", vmax=1e2)
# plot_1d_hist(pred_l1_theta_phi[2], true_l1_theta_phi[2], r"$\cos(\phi^\ast_{\ell^-})$", np.linspace(-1, 1, 61), unit="null")
# plot_2d_hist(pred_l1_theta_phi[2], true_l1_theta_phi[2], r"$\cos(\phi^\ast_{\ell^-})$", np.linspace(-1, 1, 61), log=True, unit="null", color="black", vmax=1e2)

In [ ]:
# true_l0_phi = np.arctan2(true_l0_theta_phi[1], true_l0_theta_phi[2])
# true_l1_phi = np.arctan2(true_l1_theta_phi[1], true_l1_theta_phi[2])
# pred_l0_phi = np.arctan2(pred_l0_theta_phi[1], pred_l0_theta_phi[2])
# pred_l1_phi = np.arctan2(pred_l1_theta_phi[1], pred_l1_theta_phi[2])
true_l0_phi = true_l0_theta_phi[1]
true_l1_phi = true_l1_theta_phi[1]
pred_l0_phi = pred_l0_theta_phi[1]
pred_l1_phi = pred_l1_theta_phi[1]

plot_1d_hist(pred_l0_phi / np.pi, true_l0_phi / np.pi, r"$\phi^\ast_{\ell^+}$", np.linspace(-1, 1, 61), unit="rad/$\pi$")
plot_2d_hist(pred_l0_phi / np.pi, true_l0_phi / np.pi, r"$\phi^\ast_{\ell^+}$", np.linspace(-1, 1, 61), log=True, unit="rad/$\pi$", color="black", vmax=2e2)
plot_1d_hist(pred_l1_phi / np.pi, true_l1_phi / np.pi, r"$\phi^\ast_{\ell^-}$", np.linspace(-1, 1, 61), unit="rad/$\pi$")
plot_2d_hist(pred_l1_phi / np.pi, true_l1_phi / np.pi, r"$\phi^\ast_{\ell^-}$", np.linspace(-1, 1, 61), log=True, unit="rad/$\pi$", color="black", vmax=2e2)

In [ ]:
# sum and diff of theta
sum_theta_pred = pred_l0_theta_phi[0] + pred_l1_theta_phi[0]
sum_theta_true = true_l0_theta_phi[0] + true_l1_theta_phi[0]
diff_theta_pred = pred_l0_theta_phi[0] - pred_l1_theta_phi[0]
diff_theta_true = true_l0_theta_phi[0] - true_l1_theta_phi[0]
plot_1d_hist(sum_theta_pred / np.pi, sum_theta_true / np.pi, r"$\sum_{+-}{\theta^\ast_{\ell}}$", np.linspace(0, 2, 51), unit="rad/$\pi$")
plot_2d_hist(sum_theta_pred / np.pi, sum_theta_true / np.pi, r"$\sum_{+-}{\theta^\ast_{\ell}}$", np.linspace(0, 2, 51), log=True, unit="rad/$\pi$", color="black", vmax=8e2)
plot_1d_hist(diff_theta_pred / np.pi, diff_theta_true / np.pi, r"$\Delta_{+-}\theta^\ast_{\ell}$", np.linspace(-1, 1, 61), unit="rad/$\pi$")
plot_2d_hist(diff_theta_pred / np.pi, diff_theta_true / np.pi, r"$\Delta_{+-}\theta^\ast_{\ell}$", np.linspace(-1, 1, 61), log=True, unit="rad/$\pi$", color="black", vmax=8e2)

In [ ]:
# sum and diff of phi
sum_phi_pred = sphi(pred_l0_theta_phi[1], pred_l1_theta_phi[1])
sum_phi_true = sphi(true_l0_theta_phi[1], true_l1_theta_phi[1])
diff_phi_pred = dphi(pred_l0_theta_phi[1], pred_l1_theta_phi[1])
diff_phi_true = dphi(true_l0_theta_phi[1], true_l1_theta_phi[1])
plot_1d_hist(sum_phi_pred / np.pi, sum_phi_true / np.pi, r"$\sum_{+-}{\phi^\ast_{\ell}}$", np.linspace(-1, 1, 61), unit="rad/$\pi$")
plot_2d_hist(sum_phi_pred / np.pi, sum_phi_true / np.pi, r"$\sum_{+-}{\phi^\ast_{\ell}}$", np.linspace(-1, 1, 61), log=False, unit="rad/$\pi$", color="lightgray", vmax=3e2)
plot_1d_hist(diff_phi_pred / np.pi, diff_phi_true / np.pi, r"$\Delta_{+-}\phi^\ast_{\ell}$", np.linspace(-1, 1, 61), unit="rad/$\pi$")
plot_2d_hist(diff_phi_pred / np.pi, diff_phi_true / np.pi, r"$\Delta_{+-}\phi^\ast_{\ell}$", np.linspace(-1, 1, 61), log=False, unit="rad/$\pi$", color="lightgray", vmax=3e2)

In [ ]:
# lep_p4 = np.concatenate([true_lep0_p4, true_lep1_p4], axis=1)
# true_w_p4 = np.concatenate([true_w0_p4, true_w1_p4], axis=1)
# pred_w_p4 = np.concatenate([corr_pred_w0_p4, corr_pred_w1_p4], axis=1)

# def angular_features_like_training(lep_p4, true_w_p4, pred_w_p4, device):
#     lep = torch.as_tensor(lep_p4, dtype=torch.float32, device=device)
#     true_w = torch.as_tensor(true_w_p4, dtype=torch.float32, device=device)
#     pred_w = torch.as_tensor(pred_w_p4, dtype=torch.float32, device=device)

#     true_booster = TorchBooster(lep, true_w)
#     pred_booster = TorchBooster(lep, pred_w)
#     valid = true_booster.valid_rest_frame_mask() & pred_booster.valid_rest_frame_mask()

#     true_ang = torch.stack(true_booster.lep_theta_phi_in_w_rest(), dim=-1)[valid]
#     pred_ang = torch.stack(pred_booster.lep_theta_phi_in_w_rest(), dim=-1)[valid]
#     return true_ang.cpu().numpy(), pred_ang.cpu().numpy(), valid.cpu().numpy()

# true_ang, pred_ang, angular_valid = angular_features_like_training(lep_p4, true_w_p4, pred_w_p4, device)
# print(f"Angular valid events: {angular_valid.sum()} / {len(angular_valid)}")

# # true_l0_theta_phi = (true_ang[:, 0], true_ang[:, 1], true_ang[:, 2])
# # true_l1_theta_phi = (true_ang[:, 3], true_ang[:, 4], true_ang[:, 5])
# # pred_l0_theta_phi = (pred_ang[:, 0], pred_ang[:, 1], pred_ang[:, 2])
# # pred_l1_theta_phi = (pred_ang[:, 3], pred_ang[:, 4], pred_ang[:, 5])
# true_l0_theta_phi = (true_ang[:, 0], true_ang[:, 1])
# true_l1_theta_phi = (true_ang[:, 2], true_ang[:, 3])
# pred_l0_theta_phi = (pred_ang[:, 0], pred_ang[:, 1])
# pred_l1_theta_phi = (pred_ang[:, 2], pred_ang[:, 3])
# # true_l0_phi = np.arctan2(true_l0_theta_phi[1], true_l0_theta_phi[2])
# # true_l1_phi = np.arctan2(true_l1_theta_phi[1], true_l1_theta_phi[2])
# # pred_l0_phi = np.arctan2(pred_l0_theta_phi[1], pred_l0_theta_phi[2])
# # pred_l1_phi = np.arctan2(pred_l1_theta_phi[1], pred_l1_theta_phi[2])
# true_l0_phi = true_l0_theta_phi[1]
# true_l1_phi = true_l1_theta_phi[1]
# pred_l0_phi = pred_l0_theta_phi[1]
# pred_l1_phi = pred_l1_theta_phi[1]

# plot_1d_hist(pred_l0_phi / np.pi, true_l0_phi / np.pi, r"$\phi^\ast_{\ell^+}$", np.linspace(-1, 1, 61), unit="rad/$\pi$")
# plot_2d_hist(pred_l0_phi / np.pi, true_l0_phi / np.pi, r"$\phi^\ast_{\ell^+}$", np.linspace(-1, 1, 61), log=True, unit="rad/$\pi$", color="black", vmax=1e2)
# plot_1d_hist(pred_l1_phi / np.pi, true_l1_phi / np.pi, r"$\phi^\ast_{\ell^-}$", np.linspace(-1, 1, 61), unit="rad/$\pi$")
# plot_2d_hist(pred_l1_phi / np.pi, true_l1_phi / np.pi, r"$\phi^\ast_{\ell^-}$", np.linspace(-1, 1, 61), log=True, unit="rad/$\pi$", color="black", vmax=1e2)


In [ ]:
# plot_1d_hist(pred_l0_theta_phi[0] / np.pi, true_l0_theta_phi[0] / np.pi, r"$\theta^\ast_{\ell^+}$", np.linspace(0, 1, 51), unit="rad/$\pi$")
# plot_2d_hist(pred_l0_theta_phi[0] / np.pi, true_l0_theta_phi[0] / np.pi, r"$\theta^\ast_{\ell^+}$", np.linspace(0, 1, 51), log=True, unit="rad/$\pi$", color="black", vmax=1e2)
# plot_1d_hist(pred_l1_theta_phi[0] / np.pi, true_l1_theta_phi[0] / np.pi, r"$\theta^\ast_{\ell^-}$", np.linspace(0, 1, 51), unit="rad/$\pi$")
# plot_2d_hist(pred_l1_theta_phi[0] / np.pi, true_l1_theta_phi[0] / np.pi, r"$\theta^\ast_{\ell^-}$", np.linspace(0, 1, 51), log=True, unit="rad/$\pi$", color="black", vmax=1e2)

In [ ]:
if "LOG_DIR" not in globals():
    log_dirs = sorted((RUN_DIR / "logs").glob("version_*"), key=lambda path: path.stat().st_mtime, reverse=True)
    LOG_DIR = log_dirs[0] if log_dirs else None

if LOG_DIR is None:
    print("No logs found.")
else:
    metrics_path = LOG_DIR / "metrics.csv"
    print(f"Using metrics file: {metrics_path}")
    if not metrics_path.exists():
        print(f"No metrics.csv found in {LOG_DIR}")
    else:
        df = pd.read_csv(metrics_path)
        df_clean_train = df[df["loss"].notna()] if "loss" in df.columns else pd.DataFrame()
        df_clean_val = df[df["val_loss"].notna()] if "val_loss" in df.columns else pd.DataFrame()

        if df_clean_train.empty or df_clean_val.empty:
            print("Not enough training or validation records to visualize.")
        else:
            plot_configs = [
                ("loss", "val_loss", "Adaptive Total Loss", "Loss"),
                ("static_loss", "val_static_loss", "Static Weighted Loss", "Static Loss"),
                ("huber_loss", "val_huber_loss", "Huber Loss", "Huber Loss"),
                ("neg_r2_loss", "val_neg_r2_loss", "Negative $R^2$ Loss", "Neg $R^2$ Loss"),
                ("angular_loss_mmd_loss", "val_angular_loss_mmd_loss", "Angular MMD Loss", "Angular MMD Loss"),
                ("dinu_pt_loss", "val_dinu_pt_loss", r"$p^{\nu\nu}_T$ Loss", r"$p^{\nu\nu}_T$ Loss"),
                ("higgs_mass_loss", "val_higgs_mass_loss", "Higgs Mass Loss", "Higgs Mass Loss"),
                ("w_mass_mmd0_loss", "val_w_mass_mmd0_loss", r"$W^+$ Mass MMD Loss", r"$W^+$ Mass MMD Loss"),
                ("w_mass_mmd1_loss", "val_w_mass_mmd1_loss", r"$W^-$ Mass MMD Loss", r"$W^-$ Mass MMD Loss"),
                ("w_mass_mmd_loss", "val_w_mass_mmd_loss", r"$W$ Mass MMD Loss", r"$W$ Mass MMD Loss"),
                ("w0_mass_mae_loss", "val_w0_mass_mae_loss", r"$W^+$ Mass MAE Loss", r"$W^+$ Mass MAE Loss"),
                ("w1_mass_mae_loss", "val_w1_mass_mae_loss", r"$W^-$ Mass MAE Loss", r"$W^-$ Mass MAE Loss"),
                ("w_mass_huber_loss", "val_w_mass_huber_loss", r"$W$ Mass Huber Loss", r"$W$ Mass Huber Loss"),
                ("aux_mom_mmd0_loss", "val_aux_mom_mmd0_loss", r"$W^+$ Momentum MMD Loss", r"$W^+$ Momentum MMD Loss"),
                ("aux_mom_mmd1_loss", "val_aux_mom_mmd1_loss", r"$W^-$ Momentum MMD Loss", r"$W^-$ Momentum MMD Loss"),
                ("dmet_loss", "val_dmet_loss", r"$\Delta \mathrm{MET}$ Loss", r"$\Delta \mathrm{MET}$ Loss"),
            ]
            available_configs = [
                item for item in plot_configs
                if item[0] in df_clean_train.columns
                and item[1] in df_clean_val.columns
                and not df_clean_train[item[0]].dropna().empty
                and not df_clean_val[item[1]].dropna().empty
            ]

            trainer_cfg = cfg.get("trainer", {})
            monitor_metric = trainer_cfg.get("monitor_metric", "val_static_loss")
            monitor_mode = trainer_cfg.get("monitor_mode", "min")
            if monitor_metric not in df_clean_val.columns or df_clean_val[monitor_metric].dropna().empty:
                monitor_metric = "val_loss"
            monitor_values = df_clean_val[["epoch", monitor_metric]].dropna()
            best_monitor_idx = monitor_values[monitor_metric].idxmax() if monitor_mode == "max" else monitor_values[monitor_metric].idxmin()
            best_monitor_epoch = df_clean_val.loc[best_monitor_idx, "epoch"]
            best_monitor_value = df_clean_val.loc[best_monitor_idx, monitor_metric]

            ncols = min(3, len(available_configs))
            nrows = int(np.ceil(len(available_configs) / ncols))
            fig, axes = plt.subplots(
                nrows,
                ncols,
                figsize=(5.2 * ncols, 4 * nrows),
                sharex=True,
                layout="constrained",
            )
            axes = np.atleast_1d(axes).flatten()

            skip = min(10, max(0, len(df_clean_train) - 1))

            for ax, (train_col, val_col, title, ylabel) in zip(axes, available_configs):
                train_values = df_clean_train[["epoch", train_col]].dropna().iloc[skip:]
                val_values = df_clean_val[["epoch", val_col]].dropna().iloc[skip:]

                train_line, = ax.plot(
                    train_values["epoch"],
                    train_values[train_col],
                    color="tab:blue",
                    linewidth=1.6,
                    label="Train",
                )
                val_line, = ax.plot(
                    val_values["epoch"],
                    val_values[val_col],
                    color="tab:red",
                    linewidth=1.6,
                    label="Validation",
                )
                best_line = ax.axvline(
                    best_monitor_epoch,
                    color="0.35",
                    linestyle="--",
                    linewidth=1,
                    label=f"Best epoch: {best_monitor_epoch:g}",
                )
                if val_col == monitor_metric:
                    ax.plot(best_monitor_epoch, best_monitor_value, "o", color="tab:red", markersize=5)

                ax.set_xlabel("Epoch", loc="right")
                ax.set_ylabel(ylabel, loc="top")
                ax.set_title(title, loc="right")
                ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 4), useMathText=True)

            for ax in axes[len(available_configs):]:
                ax.set_visible(False)

            # hep.atlas.label("Internal", data=False, loc=2, ax=axes[0])
            axes[0].legend(handles=[train_line, val_line, best_line], frameon=False, loc="best")
            plt.show()

            last_epoch = df_clean_train["epoch"].iloc[-1]
            final_monitor_value = df_clean_val[monitor_metric].dropna().iloc[-1]
            print(f"Monitor metric: {monitor_metric}")
            print(f"Best monitor epoch: {best_monitor_epoch} ({best_monitor_value:.6f}); final: {final_monitor_value:.6f}")

            print(f"Final Training Losses (Epoch {last_epoch}):")
            for train_col, _, title, _ in available_configs:
                value = df_clean_train[train_col].dropna().iloc[-1]
                print(f"  {title}: {value:.6f}")

            print("Final Validation Losses:")
            for _, val_col, title, _ in available_configs:
                value = df_clean_val[val_col].dropna().iloc[-1]
                print(f"  {title}: {value:.6f}")

            print("Best Validation Metrics:")
            for _, val_col, title, _ in available_configs:
                values = df_clean_val[["epoch", val_col]].dropna()
                best_idx = values[val_col].idxmin()
                print(f"  {title}: epoch {df_clean_val.loc[best_idx, 'epoch']}, {df_clean_val.loc[best_idx, val_col]:.6f}")


In [ ]:
grad_cols = sorted([col for col in df.columns if col.startswith("grad_cos/")])
if not grad_cols:
    print(f"No grad_cos columns found in {metrics_path}. Enable parameters.log_loss_gradient_cosines and rerun training.")
else:
    populated_counts = df[grad_cols].notna().sum()
    grad_cols = sorted(populated_counts[populated_counts > 0].index)
    if not grad_cols:
        print(f"Found {len(populated_counts)} grad_cos columns in {metrics_path}, but all values are empty.")
        print("This run was likely produced before epoch-level grad_cos logging, or with gradient-cosine logging disabled. Rerun training with parameters.log_loss_gradient_cosines: true.")
    else:
        total_grad_cols = [col for col in grad_cols if col.endswith("__total")]
        rest_grad_cols = [col for col in grad_cols if col.endswith("__rest")]
        pair_grad_cols = [col for col in grad_cols if col not in total_grad_cols and col not in rest_grad_cols]
        df_grad = df[df[grad_cols].notna().any(axis=1)].copy()
        x_col = "epoch" if "epoch" in df_grad.columns else "step"

        latest = df_grad[grad_cols].ffill().iloc[-1]

        if pair_grad_cols:
            heatmap = df_grad.set_index(x_col)[pair_grad_cols].ffill().T
            heatmap.index = [col.replace("grad_cos/", "").replace("__", " vs ") for col in heatmap.index]

            plt.figure(figsize=(16, max(8, 0.35 * len(pair_grad_cols))))
            plt.imshow(heatmap, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1, interpolation="nearest")
            plt.colorbar(label="Gradient cosine")
            plt.yticks(range(len(heatmap.index)), heatmap.index)
            tick_positions = np.linspace(0, len(heatmap.columns) - 1, min(10, len(heatmap.columns)), dtype=int)
            plt.xticks(tick_positions, heatmap.columns[tick_positions])
            plt.xlabel(x_col.capitalize())
            plt.ylabel("Loss pair")
            plt.title("Pairwise Similarity Over Training")
            plt.tight_layout()
            plt.show()

            pair_latest = latest[pair_grad_cols].dropna()
            loss_names = sorted({name for col in pair_grad_cols for name in col.replace("grad_cos/", "").split("__")})
            matrix = pd.DataFrame(np.nan, index=loss_names, columns=loss_names)
            for name in loss_names:
                matrix.loc[name, name] = 1.0
            for col, value in pair_latest.items():
                name_a, name_b = col.replace("grad_cos/", "").split("__")
                matrix.loc[name_a, name_b] = value
                matrix.loc[name_b, name_a] = value

            plt.figure(figsize=(10, 8))
            plt.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
            plt.colorbar(label="Gradient cosine")
            plt.xticks(range(len(loss_names)), loss_names, rotation=45, ha="right")
            plt.yticks(range(len(loss_names)), loss_names)
            plt.title(f"Pairwise Similarity Matrix ({x_col} {df_grad[x_col].iloc[-1]})")
            plt.tight_layout()
            plt.show()
        else:
            print("No populated pairwise grad_cos metrics found.")

        if total_grad_cols:
            total_heatmap = df_grad.set_index(x_col)[total_grad_cols].ffill().T
            total_heatmap.index = [
                col.replace("grad_cos/", "").replace("__total", "")
                for col in total_heatmap.index
            ]
            total_heatmap = total_heatmap.dropna(how="all")

            if total_heatmap.empty:
                print("No populated loss-vs-total grad_cos metrics found.")
            else:
                plt.figure(figsize=(16, max(8, 0.35 * len(total_heatmap))))
                plt.imshow(total_heatmap, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1, interpolation="nearest")
                plt.colorbar(label="Gradient cosine")
                plt.yticks(range(len(total_heatmap.index)), total_heatmap.index)
                tick_positions = np.linspace(0, len(total_heatmap.columns) - 1, min(10, len(total_heatmap.columns)), dtype=int)
                plt.xticks(tick_positions, total_heatmap.columns[tick_positions])
                plt.xlabel(x_col.capitalize())
                plt.ylabel("Loss")
                plt.title("Loss-vs-Total Similarity Over Training")
                plt.tight_layout()
                plt.show()
        else:
            print("No populated loss-vs-total grad_cos metrics found.")

        if rest_grad_cols:
            rest_heatmap = df_grad.set_index(x_col)[rest_grad_cols].ffill().T
            rest_heatmap.index = [
                col.replace("grad_cos/", "").replace("__rest", "")
                for col in rest_heatmap.index
            ]
            rest_heatmap = rest_heatmap.dropna(how="all")

            if rest_heatmap.empty:
                print("No populated loss-vs-rest grad_cos metrics found.")
            else:
                plt.figure(figsize=(16, max(8, 0.35 * len(rest_heatmap))))
                plt.imshow(rest_heatmap, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1, interpolation="nearest")
                plt.colorbar(label="Gradient cosine")
                plt.yticks(range(len(rest_heatmap.index)), rest_heatmap.index)
                tick_positions = np.linspace(0, len(rest_heatmap.columns) - 1, min(10, len(rest_heatmap.columns)), dtype=int)
                plt.xticks(tick_positions, rest_heatmap.columns[tick_positions])
                plt.xlabel(x_col.capitalize())
                plt.ylabel("Loss")
                plt.title("Loss-vs-Rest Similarity Over Training")
                plt.tight_layout()
                plt.show()
        else:
            print("No populated loss-vs-rest grad_cos metrics found.")

        print(f"Visualized {len(pair_grad_cols)} pairwise, {len(total_grad_cols)} total, and {len(rest_grad_cols)} rest grad_cos metrics across {len(df_grad)} logged rows.")


In [ ]:
# import shap
# import numpy as np
# import matplotlib as mpl
# from matplotlib import pyplot as plt

# mpl.rcParams.update(mpl.rcParamsDefault)

# # Feature order follows load_data.py train_obj construction.
# feature_names = [
#     "lep+ px", "lep+ py", "lep+ pz", "lep+ energy",
#     "lep- px", "lep- py", "lep- pz", "lep- energy",
#     "jet1 px", "jet1 py", "jet1 pz", "jet1 energy",
#     "jet2 px", "jet2 py", "jet2 pz", "jet2 energy",
#     "MET px", "MET py",
#     "m_ll", "delta eta ll", "delta phi ll", "delta phi ll-MET"
# ]

# output_names = [
#     "W+ px", "W+ py", "W+ pz", "W+ energy [GeV]",
#     "W- px", "W- py", "W- pz", "W- energy [GeV]",
# ]

# background_sample = torch.tensor(train_features[0:256]).float().to(device)
# samples_to_explain = torch.tensor(train_features[512:512+1024]).float().to(device)
# x_explain = samples_to_explain.detach().cpu().numpy()

# if len(feature_names) != x_explain.shape[1]:
#     raise ValueError(f"Expected {len(feature_names)} features, got {x_explain.shape[1]}")

# explainer = shap.GradientExplainer(model, background_sample)
# shap_values = explainer.shap_values(samples_to_explain, nsamples=200)

# if isinstance(shap_values, list):
#     shap_values = np.stack(shap_values, axis=-1)

# importance_by_output = np.abs(shap_values).mean(axis=0)
# overall_importance = importance_by_output.mean(axis=1)
# top_overall = np.argsort(overall_importance)[::-1][:20]

# plt.figure(figsize=(9, 7))
# plt.barh(
#     [feature_names[idx] for idx in top_overall[::-1]],
#     overall_importance[top_overall[::-1]],
# )
# plt.xlabel("Mean |SHAP value| across outputs")
# plt.title("Overall SHAP Feature Importance", fontsize=15)
# plt.tight_layout()
# plt.show()

# heatmap_features = top_overall[::-1]
# plt.figure(figsize=(10, 8))
# plt.imshow(importance_by_output[heatmap_features], aspect="auto", cmap="viridis")
# plt.colorbar(label="Mean |SHAP value|")
# plt.yticks(range(len(heatmap_features)), [feature_names[idx] for idx in heatmap_features])
# plt.xticks(
#     range(min(len(output_names), importance_by_output.shape[1])),
#     output_names[:importance_by_output.shape[1]],
#     rotation=45,
#     ha="right",
# )
# plt.title("SHAP Importance by Feature and Output", fontsize=15)
# plt.tight_layout()
# plt.show()

# for i in range(shap_values.shape[-1]):
#     values = shap_values[:, :, i]
#     importance = importance_by_output[:, i]
#     top = np.argsort(importance)[::-1][:15]
#     output_label = output_names[i] if i < len(output_names) else f"output {i}"

#     print(f"Top features for {output_label}:")
#     for rank, idx in enumerate(top, start=1):
#         print(f"  {rank:2d}. {feature_names[idx]}: {importance[idx]:.6f}")

#     shap.summary_plot(
#         values,
#         x_explain,
#         feature_names=feature_names,
#         plot_type="bar",
#         max_display=15,
#         show=False,
#     )
#     plt.title(f"SHAP Feature Importance: {output_label}", fontsize=15)
#     plt.tight_layout()
#     plt.show()

#     shap.summary_plot(
#         values,
#         x_explain,
#         feature_names=feature_names,
#         plot_type="dot",
#         max_display=15,
#         show=False,
#     )
#     plt.title(f"SHAP Beeswarm: {output_label}", fontsize=15)
#     plt.tight_layout()
#     plt.show()